# Data preview + ML feasibility (futures return as response)

Goal: inspect panel quality and try baseline models where `futures_return` is the response and all other variables are features.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data/processed/monthly_panel.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Run scripts/monthly_pipeline.ipynb first to generate data/processed/monthly_panel.csv")

panel = pd.read_csv(DATA_PATH)
panel["date"] = pd.to_datetime(panel["date"])
panel = panel.sort_values(["commodity", "date"]).reset_index(drop=True)

print(panel.shape)
panel.head()


(732, 17)


,date,futures_price,futures_return,commodity,price_source,usd_index,interest_rate,vix,enso_oni,disaster_event_count,disaster_storm_count,disaster_fire_count,disaster_flood_count,temperature_anomaly,precipitation_anomaly,drought_index,extreme_heat_events
0,2006-01-31,218.75,0.013809,corn,ZC=F,100.000005,4.24,12.036000,-0.85,436.0,0.0,370.0,0.0,NaN,NaN,NaN,NaN
1,2006-02-28,228.00,0.041416,corn,ZC=F,100.211170,4.43,12.471053,-0.77,44.0,0.0,3.0,0.0,NaN,NaN,NaN,NaN
2,2006-03-31,236.00,0.034486,corn,ZC=F,100.428087,4.51,11.693913,-0.57,69.0,0.0,2.0,0.0,NaN,NaN,NaN,NaN
3,2006-04-30,238.25,0.009489,corn,ZC=F,99.743480,4.60,11.847368,-0.37,37.0,0.0,5.0,0.0,NaN,NaN,NaN,NaN
4,2006-05-31,251.25,0.053128,corn,ZC=F,97.511774,4.72,14.454545,-0.14,31.0,0.0,2.0,0.0,NaN,NaN,NaN,NaN


In [2]:
print("Columns:", panel.columns.tolist())
print("\nMissing values:")
print(panel.isna().mean().sort_values(ascending=False).head(15))

print("\nDate range:", panel["date"].min().date(), "->", panel["date"].max().date())
print("Commodities:", panel["commodity"].unique())


Columns: ['date', 'futures_price', 'futures_return', 'commodity', 'price_source', 'usd_index', 'interest_rate', 'vix', 'enso_oni', 'disaster_event_count', 'disaster_storm_count', 'disaster_fire_count', 'disaster_flood_count', 'temperature_anomaly', 'precipitation_anomaly', 'drought_index', 'extreme_heat_events']

Missing values:
temperature_anomaly      1.000000
drought_index            1.000000
precipitation_anomaly    1.000000
extreme_heat_events      1.000000
disaster_event_count     0.008197
enso_oni                 0.008197
disaster_storm_count     0.008197
disaster_fire_count      0.008197
disaster_flood_count     0.008197
interest_rate            0.004098
date                     0.000000
vix                      0.000000
usd_index                0.000000
futures_return           0.000000
futures_price            0.000000
dtype: float64

Date range: 2006-01-31 -> 2026-04-30
Commodities: <StringArray>
['corn', 'soybean', 'wheat']
Length: 3, dtype: str


In [3]:
# More robust feature engineering for low-signal monthly return prediction
model_df = panel.copy()

# Predict next-month return, so every predictor is based on information available at t.
model_df["target_next_return"] = model_df.groupby("commodity")["futures_return"].shift(-1)

base_lag_cols = [
    "futures_return",
    "futures_price",
    "usd_index",
    "interest_rate",
    "vix",
    "enso_oni",
    "disaster_event_count",
    "disaster_storm_count",
    "disaster_fire_count",
    "disaster_flood_count",
]
base_lag_cols = [c for c in base_lag_cols if c in model_df.columns]

for col in base_lag_cols:
    g = model_df.groupby("commodity")[col]
    for lag in [1, 2, 3, 6, 12]:
        model_df[f"{col}_lag{lag}"] = g.shift(lag)

# Rolling features from past returns for momentum / volatility structure
ret_group = model_df.groupby("commodity")["futures_return"]
for win in [3, 6, 12]:
    model_df[f"ret_mean_{win}"] = ret_group.transform(lambda s: s.shift(1).rolling(win).mean())
    model_df[f"ret_std_{win}"] = ret_group.transform(lambda s: s.shift(1).rolling(win).std())

model_df["month"] = model_df["date"].dt.month.astype(str)

target_col = "target_next_return"
model_df = model_df.dropna(subset=[target_col]).copy()

# Remove very sparse columns (optional climate inputs may be mostly missing)
missing_rate = model_df.isna().mean()
sparse_cols = [c for c in model_df.columns if c not in [target_col, "date"] and missing_rate[c] > 0.90]
if sparse_cols:
    print("Dropping sparse columns:", sparse_cols)
    model_df = model_df.drop(columns=sparse_cols)

feature_cols = [c for c in model_df.columns if c not in [target_col, "date"]]

# Walk-forward holdout: final 24 months across all commodities
unique_months = np.sort(model_df["date"].unique())
if len(unique_months) < 36:
    raise ValueError("Need at least 36 monthly points for train/test split")

cutoff = unique_months[-24]
train_df = model_df[model_df["date"] < cutoff].copy()
test_df = model_df[model_df["date"] >= cutoff].copy()

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
    ]
)

models = {
    "naive_lag1": None,
    "ridge": Ridge(alpha=2.5),
    "elastic_net": ElasticNet(alpha=0.0025, l1_ratio=0.25, random_state=42, max_iter=10000),
    "random_forest": RandomForestRegressor(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    ),
    "hist_gbdt": HistGradientBoostingRegressor(
        max_depth=4,
        learning_rate=0.03,
        max_iter=500,
        random_state=42,
    ),
}

results = []
for name, estimator in models.items():
    if name == "naive_lag1":
        # commodity-wise naive benchmark: use most recent observed return
        naive_col = "futures_return_lag1"
        if naive_col not in X_test.columns:
            continue
        pred = X_test[naive_col].fillna(y_train.mean()).to_numpy()
    else:
        pipe = Pipeline([
            ("preprocess", preprocess),
            ("model", estimator),
        ])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)

    rmse = mean_squared_error(y_test, pred) ** 0.5
    results.append({
        "model": name,
        "mae": mean_absolute_error(y_test, pred),
        "rmse": rmse,
        "r2": r2_score(y_test, pred),
    })

results_df = pd.DataFrame(results).sort_values("rmse")
results_df


Dropping sparse columns: ['temperature_anomaly', 'precipitation_anomaly', 'drought_index', 'extreme_heat_events']


,model,mae,rmse,r2
3,random_forest,0.042642,0.056331,0.049237
2,elastic_net,0.046148,0.059874,-0.074100
1,ridge,0.046793,0.060732,-0.105112
4,hist_gbdt,0.047255,0.062093,-0.155182
0,naive_lag1,0.070942,0.092511,-1.564214


In [4]:

# quick look at target behavior by commodity
summary = (
    panel.groupby("commodity")["futures_return"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(4)
)
summary


,count,mean,std,min,max
commodity,,,,,
corn,244,0.0030,0.0864,-0.3084,0.2712
soybean,244,0.0027,0.0710,-0.2427,0.1807
wheat,244,0.0023,0.0907,-0.2910,0.3530


## Interpretation checklist
- If `r2` is near 0 or negative, current features provide weak predictive power for monthly return.
- Disaster features may still help as interaction terms (e.g., with commodity and season).
- Next iteration: rolling-window validation, add lagged macro/disaster variables, and compare classification of return sign (+/-).